# Raw-Layer Mapping

이 노트북은 raw 레이어 대시보드를 만들기 전에, **현재 실제 layer에 쓰이는 raw 데이터**만 먼저 정리하고 시각화한다.

핵심 질문:

- 어떤 raw 데이터가 현재 layer로 변환되는가?
- 그 raw 데이터는 어떤 source type인가?
- 어떤 layer와 score에 연결되는가?

## 용어 정리

- **raw 데이터**: CSV/XLSX, OSM, Kakao, 공공데이터 API처럼 아직 서비스용 layer로 가공되기 전의 원천 데이터
- **layer**: raw 데이터를 정제해서 경로 추천이나 score 계산에 쓰기 좋게 만든 서비스용 공간 데이터
- **V1 데이터 범위**: `analysis/data-governance/README.md`에 현재 실행 코드 기준의 raw, layer, score 연결 상태를 기록한다.
- **active**: 현재 `src/data/data_collector.py` 실행 흐름에서 실제 layer/score에 반영되는 상태

## 현재 실제 layer에 쓰이는 raw 데이터

아래 표는 inactive, draft, raw-only 데이터를 제외하고, 현재 layer/score에 반영되는 raw만 남긴 것이다.

참고: `서울시 자치구별 도보 네트워크 공간정보.csv`는 `walk_nodes` / `walk_edges`를 만드는 기반 데이터지만, score layer는 아니므로 이 시각화에서는 제외한다.

In [10]:
import pandas as pd

active_raw = pd.DataFrame([
    {"raw": "전국어린이보호구역표준데이터.csv", "source_type": "CSV", "raw_table": "csv_raw", "collector": "ChildCollector", "layer": "child_layer", "score": "child_score"},
    {"raw": "어린이놀이시설 API", "source_type": "Public API", "raw_table": "public_raw", "collector": "ChildCollector", "layer": "child_layer", "score": "child_score"},
    {"raw": "전국스마트가로등표준데이터.csv", "source_type": "CSV", "raw_table": "csv_raw", "collector": "SafetyCollector", "layer": "safety_layer", "score": "safety_score"},
    {"raw": "서울시CCTV정보.xlsx", "source_type": "XLSX", "raw_table": "csv_raw", "collector": "SafetyCollector", "layer": "safety_layer", "score": "safety_score"},
    {"raw": "보행자 교통사고 다발지점 API", "source_type": "Public API", "raw_table": "public_raw", "collector": "SafetyCollector.update_accident", "layer": "safety_layer", "score": "safety_score"},
    {"raw": "OSM 녹지 태그 데이터", "source_type": "OSM", "raw_table": "osm_raw", "collector": "NatureCollector", "layer": "nature_layer", "score": "nature_score"},
    {"raw": "TourAPI 관광지/문화시설", "source_type": "Public API", "raw_table": "public_raw", "collector": "LandmarkCollector", "layer": "landmark_layer", "score": "landmark_score"},
    {"raw": "전국실외운동기구설치정보 API", "source_type": "Public API", "raw_table": "public_raw", "collector": "RunningCourseCollector.update_outdoor_exercise", "layer": "running_layer", "score": "running_score"},
])

active_raw

,raw,source_type,raw_table,collector,layer,score
0,전국어린이보호구역표준데이터.csv,CSV,csv_raw,ChildCollector,child_layer,child_score
1,어린이놀이시설 API,Public API,public_raw,ChildCollector,child_layer,child_score
2,전국스마트가로등표준데이터.csv,CSV,csv_raw,SafetyCollector,safety_layer,safety_score
3,서울시CCTV정보.xlsx,XLSX,csv_raw,SafetyCollector,safety_layer,safety_score
4,보행자 교통사고 다발지점 API,Public API,public_raw,SafetyCollector.update_accident,safety_layer,safety_score
5,OSM 녹지 태그 데이터,OSM,osm_raw,NatureCollector,nature_layer,nature_score
6,TourAPI 관광지/문화시설,Public API,public_raw,LandmarkCollector,landmark_layer,landmark_score
7,전국실외운동기구설치정보 API,Public API,public_raw,RunningCourseCollector.update_outdoor_exercise,running_layer,running_score


## Layer별 사용 raw 개수

현재 실제 layer에 연결된 raw가 layer별로 몇 개인지 확인한다.

In [11]:
layer_counts = active_raw.groupby("layer").size().reset_index(name="raw_count")
layer_counts

,layer,raw_count
0,child_layer,2
1,landmark_layer,1
2,nature_layer,1
3,running_layer,1
4,safety_layer,3


In [ ]:
ax = layer_counts.plot.bar(x="layer", y="raw_count", legend=False, figsize=(8, 4), title="Active raw count by layer")
ax.set_xlabel("Layer")
ax.set_ylabel("Raw count")

## Source type별 사용 raw 개수

현재 layer에 쓰이는 raw가 CSV/XLSX, OSM, Public API 중 어디에 많이 의존하는지 확인한다.

In [12]:
source_counts = active_raw.groupby("source_type").size().reset_index(name="raw_count")
source_counts

,source_type,raw_count
0,CSV,2
1,OSM,1
2,Public API,4
3,XLSX,1


In [ ]:
ax = source_counts.plot.bar(x="source_type", y="raw_count", legend=False, figsize=(8, 4), title="Active raw count by source type")
ax.set_xlabel("Source type")
ax.set_ylabel("Raw count")

## 현재 대시보드 본문에서 제외하는 데이터

- `running_park`, `bike_road`, `river_geojson`: `RunningCourseCollector.save()`가 현재 기본 파이프라인에서 비활성이다.
- `street_tree`, `seoul_trail`: `approved=false`인 draft 데이터다.
- `kakao_category_places`: `kakao_raw`에 적재될 수 있지만 현재 layer collector가 사용하지 않는다.
- `toilet`, `bus_stop`, `commercial`, `bike_road_seoul`: raw loader는 있으나 현재 layer 연결 근거가 없다.

따라서 raw 대시보드는 먼저 active raw를 중심으로 만들고, 제외 데이터는 별도 참고 표나 경고 섹션으로 분리하는 것이 적절하다.